# Linear Regression from Scratch — A Hands-On Tutorial

*Build, train, and evaluate a linear-regression model using only NumPy — then see it learn, step by step.*

---

## Overview

This tutorial turns a compact `linear_regression.py` script into a complete, visual learning experience.

- **What it teaches:** how linear regression works, how gradient descent trains it, and how to evaluate a regression model.
- **Who it's for:** beginners and intermediate learners who know a little Python and want to understand the math *and* the code behind the simplest ML model.
- **ML task type:** **Regression** — predicting a *continuous number* (not a category).

### Learning outcomes
By the end you will be able to:
1. Explain the line equation $\hat{y} = wx + b$ and what $w$ and $b$ mean.
2. Describe how **gradient descent** adjusts the parameters to reduce error.
3. Evaluate a regression model with **MSE** and **R²**.
4. Read an **actual-vs-predicted** plot and a **residual** plot to judge model quality.

### What you will build
A from-scratch `LinearRegression` class (no scikit-learn model) that fits a straight line to data, plus a set of charts — including an **animation of the line being learned** iteration by iteration.

## 1. The concept in plain language

**Linear regression** fits the *best straight line* through your data so you can predict a **continuous number**.

The model is just the equation of a line:

$$\hat{y} = w \cdot x + b$$

- $x$ — the input **feature** (e.g. size of a house)
- $\hat{y}$ — the **prediction** (e.g. its price)
- $w$ — the **weight** (slope): how much $y$ changes when $x$ increases by 1
- $b$ — the **bias** (intercept): the value of $y$ when $x = 0$

### Real-world example
Predicting **house prices** from size: bigger houses cost more, roughly along a straight line. Give the model a size, it returns a price.

### Analogy
Imagine drawing a line through a scatter of points so the line sits *as close as possible* to all of them. "As close as possible" means the total squared vertical gap between the points and the line is the smallest it can be. Linear regression finds that line automatically.

### How it learns: gradient descent
The model starts with a flat line ($w=0, b=0$) and repeatedly nudges $w$ and $b$ in the direction that reduces the error — like walking downhill on an error surface until you reach the bottom. Each nudge is controlled by the **learning rate**.

### Key terms
- **Feature / target** — the input ($x$) and the thing we predict ($y$).
- **Loss (MSE)** — average squared error; what we minimize.
- **Learning rate** — step size for each update; too big overshoots, too small is slow.
- **Epoch / iteration** — one full update of the parameters.

## 2. The machine-learning workflow

Almost every supervised-learning project follows the same path. Here is how this notebook maps onto it:

```mermaid
flowchart LR
    A[Generate / load data] --> B[Explore the data]
    B --> C[Train / test split]
    C --> D[Train model - gradient descent]
    D --> E[Predict on test set]
    E --> F[Evaluate - MSE & R squared]
    F --> G[Visualise fit & residuals]
    G --> H[Interpret & improve]
```

| Step | What it means | Where in this notebook |
|------|---------------|------------------------|
| Data loading | Get numbers to learn from | Section 5 |
| Data exploration (EDA) | Look at shape, stats, distributions | Section 6 |
| Train/test split | Hold out data for honest evaluation | Section 5 |
| Model training | Fit the line with gradient descent | Section 7 |
| Prediction | Apply the line to unseen inputs | Section 7 |
| Evaluation | Measure error with MSE and R² | Section 8 |
| Interpretation | Read the fit, residuals, and animation | Sections 9–10 |

**Note on scaling:** this toy dataset has a single, already well-scaled feature, so we skip feature scaling. For real multi-feature data you would typically **standardize** features first, because gradient descent converges much faster when all features share a similar range.

## 0. Setup

We use a small, standard stack:

- **NumPy** — the array math behind the model (dot products, gradients).
- **pandas** — to hold the data as a table for quick exploration.
- **Matplotlib / seaborn** — for the charts.
- **scikit-learn** — *only* to generate a toy dataset and split it. **The linear-regression model itself is written from scratch.**

> If a package is missing, install it once with:
> `pip install numpy pandas matplotlib seaborn scikit-learn`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn import datasets

# Nicer default styling for all plots
sns.set_theme(style="whitegrid")
%matplotlib inline

## 3. Evaluation metrics (defined first so we can reuse them)

Before the model, we define the two numbers we'll use to judge it:

- **Mean Squared Error (MSE)** — the average of the squared gaps between predictions and truth. **Lower is better**; 0 means perfect. Squaring punishes big misses more and keeps the sign from cancelling out.
- **R² (coefficient of determination)** — the fraction of the target's variance the model explains. **1.0 is perfect**, 0 means "no better than predicting the mean", and negative means "worse than the mean".

$$\text{MSE} = \frac{1}{n}\sum_i (y_i - \hat{y}_i)^2 \qquad R^2 = 1 - \frac{\sum_i (y_i - \hat{y}_i)^2}{\sum_i (y_i - \bar{y})^2}$$

> **Refactor note:** the original script computed R² from a correlation coefficient (`np.corrcoef(...)**2`). Here it uses the **standard definition** $1 - SS_{res}/SS_{tot}$. They agree for a good fit, but the standard formula is more general (it can correctly go negative for a bad model), so it's the better teaching version.

In [ ]:
def r2_score(y_true, y_pred):
    """
    Calculate the R-squared (coefficient of determination) score.

    Parameters:
    y_true (array-like): True target values.
    y_pred (array-like): Predicted target values.

    Returns:
    float: R-squared score.
    """
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    
    r2 = 1 - (ss_res / ss_tot)
    
    return r2

def mean_squared_error(y_true, y_pred):
    """
    Calculate the Mean Squared Error (MSE).

    Parameters:
    y_true (array-like): True target values.
    y_pred (array-like): Predicted target values.

    Returns:
    float: Mean Squared Error.
    """
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    mse = np.mean((y_true - y_pred) ** 2)
    
    return mse

**What just happened:** we defined `mean_squared_error` and `r2_score` as plain functions. They take the true values and the predictions and return a single number each. We'll call them after training to score the model. Notice both convert inputs with `np.array(...)` so they work with lists or NumPy arrays.

## 4. The model — Linear Regression from scratch

The class below is the whole model. It has the same three methods as scikit-learn's estimators:

- **`fit(X, y)`** — learns `weights` (w) and `bias` (b) by **gradient descent**.
- **`predict(X)`** — applies the learned line: `X · w + b`.

### How gradient descent works here
Each iteration does four things:
1. **Predict** with the current line: `y_pred = X·w + b`.
2. **Measure the error**: `y_pred − y`.
3. **Compute gradients** — the direction that increases error:
   $$dw = \tfrac{1}{n}X^{T}(\hat{y}-y), \qquad db = \tfrac{1}{n}\sum(\hat{y}-y)$$
4. **Step downhill**: subtract `learning_rate × gradient` from `w` and `b`.

Repeat `n_iterations` times and the line settles into the best fit.

**Parameters to notice:**
- `learning_rate` — step size. Too large → the line overshoots and diverges; too small → very slow learning.
- `n_iterations` — how many update steps to run.

In [ ]:
class LinearRegression:
    def __init__(self, n_iterations=1000, learning_rate=0.01):
        self.weights = None
        self.bias = None
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations

    def fit(self, X, y):
        """
        Fit the linear regression model to the training data.

        Parameters:
        X (array-like): Training features.
        y (array-like): Training target values.
        """
        X = np.array(X)
        y = np.array(y)
        
        n_samples, n_features = X.shape
        # Initialize weights and bias
        self.weights = np.zeros(n_features)
        self.bias = 0

        # Gradient Descent parameters
        for _ in range(self.n_iterations):  # Number of iterations
            y_pred = np.dot(X, self.weights) + self.bias
            # Compute gradients
            dw = (1 / n_samples) * np.dot(X.T, (y_pred - y))
            db = (1 / n_samples) * np.sum(y_pred - y)

            # Update weights and bias
            self.weights -= self.learning_rate * dw
            self.bias -= self.learning_rate * db

    def predict(self, X):
        """
        Predict target values for given input features.

        Parameters:
        X (array-like): Input features.

        Returns:
        array-like: Predicted target values.
        """
        X = np.array(X)
        return np.dot(X, self.weights) + self.bias

**What just happened:** we defined the `LinearRegression` class but haven't trained anything yet — a class definition only creates the *blueprint*. `weights` and `bias` are `None` until we call `fit`. Because everything is plain NumPy, you can read exactly how the model learns, with no hidden library magic.

## 5. The data + train/test split

We generate a simple 1-feature regression dataset with scikit-learn's `make_regression`. Using a synthetic dataset means **there is no file to download** and the true relationship is genuinely linear (plus noise), which is perfect for learning.

- `n_samples=100` — 100 points.
- `n_features=1` — a single input, so we can plot everything in 2D.
- `noise=20` — scatter added around the line, so the fit isn't trivial.
- `random_state=42` — makes the data reproducible.

We then hold out **20%** as a **test set**. The model learns only from the training set; the test set is used to check how well it generalizes to unseen data.

In [ ]:
X, y = datasets.make_regression(n_samples=100, n_features=1, noise=20, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1234)


**What just happened:** `make_regression` returned a feature matrix `X` (shape `100 × 1`) and target `y`. `train_test_split` shuffled and carved off 20% for testing. From now on, `X_train`/`y_train` are for learning and `X_test`/`y_test` are the untouched exam the model must pass.

## 6. Exploratory data analysis (EDA)

Before training, we *look* at the data: its shape, whether values are missing or duplicated, its summary statistics, and — most importantly for regression — the **shape of the relationship** between the feature and the target.

In [ ]:
# Put the data in a DataFrame for easy inspection
df = pd.DataFrame({"feature": X.ravel(), "target": y})

print("Shape (rows, cols):", df.shape)
print("Missing values:", df.isna().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

display(df.head())
display(df.describe())

The table confirms the basics: 100 rows, 2 columns, **no missing values and no duplicates**, so no data cleaning is needed here (a luxury of synthetic data — real datasets usually need work at this step). The `describe()` summary shows the range and spread of both the feature and the target.

In [ ]:
# Visualise the target distribution and the feature-vs-target relationship
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: how the target values are spread out
sns.histplot(df["target"], kde=True, color="steelblue", ax=axes[0])
axes[0].set_title("Target distribution")
axes[0].set_xlabel("target (y)")
axes[0].set_ylabel("count")

# Right: the relationship we want the model to learn
sns.scatterplot(data=df, x="feature", y="target", color="darkorange", ax=axes[1])
axes[1].set_title("Feature vs target (the trend to learn)")
axes[1].set_xlabel("feature (X)")
axes[1].set_ylabel("target (y)")

plt.tight_layout()
plt.show()

**Key insight:** the scatter on the right shows a clear upward trend — as the feature increases, the target increases roughly linearly. That straight-line pattern is exactly what linear regression is designed to capture, so we can expect a good fit. The target histogram (left) is roughly bell-shaped and centered near zero, which is typical for `make_regression` output.

*Why seaborn here?* For quick statistical views — a histogram with a smooth **KDE** curve and a clean scatter — seaborn produces publication-quality results in one line each, which is clearer for teaching than hand-built Matplotlib. We keep Matplotlib for the custom model-fit and animation plots later, where we need fine control.

## 7. Train the model

Now we create the model, call `fit` on the **training data** (this runs 1000 gradient-descent steps), and use `predict` on the **test data** to get predictions for points the model never saw.

In [ ]:
reg_model = LinearRegression(n_iterations=1000, learning_rate=0.01)
reg_model.fit(X_train, y_train)
predictions = reg_model.predict(X_test)     


**What just happened:** `fit` ran gradient descent and stored the learned `weights` and `bias` inside `reg_model`. `predict` then turned the test inputs into predicted numbers stored in `predictions`. Nothing is printed yet — we score the result in the next section.

In [ ]:
mse = mean_squared_error(y_test, predictions)
r2 = r2_score(y_test, predictions)
print(f"Mean Squared Error: {mse}")
print(f"R-squared Score: {r2}")
    

## 8. Evaluate the model

**How to read these two numbers:**
- **MSE** is in *squared target units*, so on its own it's hard to interpret — it's most useful for *comparing* models on the same data (lower = better).
- **R²** is the intuitive one: close to **1.0** means the line explains almost all the variation in the target. For this clean, linear dataset you should see a high R² (around 0.9+), confirming a strong fit.

## 9. Visualising the fit

Numbers tell us *how well*; charts tell us *how*. First we draw the learned line over the data. A good fit should pass through the middle of the point cloud, with points scattered evenly above and below it.

In [ ]:
y_pred_line = reg_model.predict(X)   # the fitted straight line evaluated over all X

cmap = plt.get_cmap('viridis')
plt.figure(figsize=(8, 6))
plt.scatter(X_train, y_train, color=cmap(0.9), s=12, label="train data")
plt.scatter(X_test, y_test, color=cmap(0.5), s=12, label="test data")
plt.plot(X, y_pred_line, color='black', linewidth=2, label="model prediction")
plt.xlabel("feature (X)")
plt.ylabel("target (y)")
plt.title("Linear regression fit: the learned line through the data")
plt.legend()
plt.show()

The black line is the model. Notice it runs through the center of both the training (dark) and test (lighter) points — a sign the model generalizes, not just memorizes.

### Actual vs predicted
Another way to judge a regressor: plot the **true** test values against the **predicted** ones. Perfect predictions would all fall on the diagonal line.

In [ ]:
# Actual vs predicted: perfect predictions would land on the diagonal
plt.figure(figsize=(6, 6))
plt.scatter(y_test, predictions, color="teal", alpha=0.8)

lims = [min(y_test.min(), predictions.min()), max(y_test.max(), predictions.max())]
plt.plot(lims, lims, "r--", linewidth=2, label="perfect prediction")

plt.xlabel("actual value")
plt.ylabel("predicted value")
plt.title("Actual vs Predicted (test set)")
plt.legend()
plt.axis("equal")
plt.show()

**Key insight:** the closer the points hug the red dashed line, the better the predictions. Points **above** the line are under-predictions; **below** are over-predictions. Tightly clustered points along the diagonal mean a high R².

### Residual plot
A **residual** is `actual − predicted` — the error left over for each point. Plotting residuals against the predictions reveals whether the model missed any structure.

In [ ]:
# Residual plot: the leftover error for each test prediction
residuals = y_test - predictions

plt.figure(figsize=(8, 4.5))
plt.scatter(predictions, residuals, color="purple", alpha=0.8)
plt.axhline(0, color="red", linestyle="--", linewidth=2)
plt.xlabel("predicted value")
plt.ylabel("residual (actual − predicted)")
plt.title("Residual plot — should look like random noise around 0")
plt.show()

**Key insight:** a good linear model leaves residuals that look like **random noise around the red zero line**, with no pattern. If you instead saw a **curve** (a U or arch), it would mean the true relationship is non-linear and a straight line is missing structure. Here the residuals should look patternless — evidence the linear model is appropriate.

## 10. Animation — watch gradient descent learn the line

This is the highlight. We re-run gradient descent and **record the line after every step**, then play it back as an inline animation (built with `matplotlib.animation`, shown via `to_jshtml()` so it needs no extra tools). Press ▶ to watch the line rotate and slide until it fits the data.

In [ ]:
from matplotlib import animation
from IPython.display import HTML

# Re-run gradient descent manually, recording the (w, b) line after each step
Xtr = np.array(X_train).ravel()
ytr = np.array(y_train)
n = len(Xtr)
w, b = 0.0, 0.0
lr = 0.05          # a slightly larger step so convergence is visible in few frames
history = []
for _ in range(40):
    y_pred = Xtr * w + b
    dw = (1 / n) * np.dot(Xtr, (y_pred - ytr))
    db = (1 / n) * np.sum(y_pred - ytr)
    w -= lr * dw
    b -= lr * db
    history.append((w, b))

line_x = np.linspace(X.min(), X.max(), 100)
fig, ax = plt.subplots(figsize=(7, 5))

def draw_step(i):
    ax.clear()
    ax.scatter(X_train, y_train, s=15, color="steelblue", label="train data")
    w_i, b_i = history[i]
    ax.plot(line_x, line_x * w_i + b_i, color="black", lw=2.5, label="current line")
    ax.set_title(f"Gradient descent — step {i + 1}/{len(history)}   (w={w_i:.1f}, b={b_i:.1f})")
    ax.set_xlabel("feature (X)")
    ax.set_ylabel("target (y)")
    ax.legend(loc="upper left")

anim = animation.FuncAnimation(fig, draw_step, frames=len(history), interval=200)
plt.close(fig)  # prevent a duplicate static figure
HTML(anim.to_jshtml())

**What to notice:** the line starts flat (`w≈0, b≈0`) and *rotates and shifts* into place as gradient descent runs. The biggest improvements happen in the first few steps, then changes get smaller as it closes in on the best fit. This is exactly the loop inside `fit` — the animation just makes each update visible.

**Learning objective:** connect the abstract "gradient descent minimizes error" to a concrete picture of a line settling onto the data.

## 11. Key takeaways

- Linear regression fits $\hat{y} = wx + b$ — the best straight line through the data.
- **Gradient descent** learns `w` and `b` by repeatedly stepping downhill on the error.
- **MSE** measures error (lower is better); **R²** measures explained variance (closer to 1 is better).
- Always check the **fit plot** and **residual plot**, not just the metrics — residuals should look like random noise around zero.
- The whole model is a few lines of NumPy: prediction is one dot product, learning is one gradient step repeated.

## 12. Common mistakes and troubleshooting

| Problem | Cause | Fix |
|--------|-------|-----|
| Loss becomes `nan` or explodes | Learning rate too high → gradient descent diverges | Lower `learning_rate` (e.g. 0.01) |
| Model barely learns | Learning rate too small or too few iterations | Increase `learning_rate` or `n_iterations` |
| Slow convergence on real data | Features on very different scales | **Standardize** features before training |
| Great train score, poor test score | Overfitting (less likely for a straight line) | Get more data / simplify / regularize |
| Curved residual pattern | The true relationship isn't linear | Add polynomial features or use a non-linear model |

## 13. Exercises

1. **Learning rate.** Re-run training with `learning_rate=0.1` and `0.001`. What happens to the fit and to R²?
2. **Noise level.** Regenerate the data with `noise=5` and `noise=50`. How does the residual spread change?
3. **Loss curve.** Modify `fit` to store the MSE each iteration, then plot it — you should see it fall and flatten.
4. **More iterations.** Does `n_iterations=100` under-fit? Does `5000` help or just cost time?
5. **Compare.** Fit scikit-learn's `LinearRegression` on the same data and compare `weights`/`bias` and R².

## 14. Next steps and improvements

- **Compare with scikit-learn's `LinearRegression`** (which solves it exactly) to check your from-scratch result.
- **Add feature scaling** (`StandardScaler`) and try multi-feature data — watch how it speeds up convergence.
- **Track the loss** each iteration and plot a **learning curve** to see training progress numerically.
- **Try regularization** (Ridge/Lasso) to handle many correlated features.
- **Tune the learning rate** and iterations to see the speed-vs-stability trade-off.

---
*Built from the `linear_regression.py` "ML from scratch" implementation. The model's gradient-descent logic is preserved; the tutorial structure, EDA, evaluation visuals, animation, and explanations were added for self-study. R² was upgraded to the standard $1 - SS_{res}/SS_{tot}$ definition.*